In [56]:
# 风险及免责提示：该策略由聚宽用户在聚宽社区分享，仅供学习交流使用。
# 原文一般包含策略说明，如有疑问请到原文和作者交流讨论。
# 原文网址：https://www.joinquant.com/view/community/detail/38296
# 标题：FoF, all in
# 作者：Gyro

import numpy as np
import pandas as pd
import datetime as dt
from jqdata import *
from jqfactor import *
from six import *

In [57]:
# set option
pd.set_option('display.width', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [58]:
# parameter
index = '000300.XSHG' # market index
mday = 21
nday = 241
rday = sqrt(nday)

In [59]:
# last day
h = history(1, '1d', 'close', index)
dt_last = h.index[0].date()
dt_last

datetime.date(2022, 6, 24)

In [60]:
# all fund
all_fund = get_all_securities('fund', dt_last)
len(all_fund)

1198

In [61]:
all_fund.head()

,display_name,name,start_date,end_date,type
159001.XSHE,货币ETF,BZJ,2014-10-20,2200-01-01,etf
159003.XSHE,招商快线ETF,ZSKX,2014-10-20,2200-01-01,etf
159005.XSHE,汇添富快钱ETF,TFKQ,2015-01-13,2200-01-01,etf
159601.XSHE,A50ETF,A50ETF,2021-11-08,2200-01-01,etf
159602.XSHE,中国A50ETF,ZGA50ETF,2021-11-08,2200-01-01,etf


In [62]:
# filter, on list 1-year
dt_1y = dt_last - dt.timedelta(days=365)
all_fund = all_fund[all_fund.start_date < dt_1y]
funds = all_fund.index.tolist()
len(funds)

960

In [63]:
# shares
ff = finance.run_query(query(
        finance.FUND_SHARE_DAILY
    ).filter(
        finance.FUND_SHARE_DAILY.code.in_(funds),
        finance.FUND_SHARE_DAILY.date==dt_last,
    )).set_index('code')
shares = ff.shares
shares.head()

code
159001.XSHE    25539800
159003.XSHE     3314700
159005.XSHE      382600
159702.XSHE    18931000
159703.XSHE    56581500
Name: shares, dtype: int64

In [64]:
#  filter, size
price = history(1, '1d', 'close', funds).iloc[0]
value = price*shares
value = value[value > 1e8].dropna()
funds = value.index.tolist()
len(funds)

443

In [65]:
# filter, liquidity
h = history(mday, '1d', 'money', funds+[index])
hm = h.min()
hx = 1e-6*h[index].mean()
funds = [s for s in funds if hm[s] > hx]
len(funds), hx/10000

(353, 30.30354752809524)

In [66]:
#  history return
h = history(nday, '1d', 'close', funds+[index])
r = np.log(h).diff()[1:]
rx = r[index]

In [67]:
# filter, volatility
V = 100*rday*r.std()
Vx = V[index]
V = V[V > 1.0]
V = V[V < Vx]
V = V.sort_values()
funds = V.index.tolist()
len(funds)

78

In [68]:
# weight
w = 1.0/V
weight = 0.95*w/w.sum()
funds = weight.index.tolist()

In [69]:
# report
df = pd.DataFrame(index=funds)
df['Value'] = 1e-8*value[funds]
df['Volatility'] = V
df['Weight'] = 100*weight
df['Name'] = all_fund.display_name[funds]
df

,Value,Volatility,Weight,Name
511030.XSHG,52.554836,1.099327,8.206155,公司债
161716.XSHE,11.390424,1.284193,7.024840,招商双债LOF
511010.XSHG,8.059388,1.749789,5.155621,国债ETF
511220.XSHG,12.521523,1.825777,4.941046,城投ETF
511060.XSHG,5.603295,1.868562,4.827910,5年地债
161911.XSHE,1.395983,1.915922,4.708566,万家强化收益定开
161115.XSHE,3.183241,1.933686,4.665311,易基岁丰添利LOF
511260.XSHG,7.458948,2.097523,4.300905,十年国债
511270.XSHG,7.059673,2.115388,4.264582,10年地债
159816.XSHE,16.817444,2.180358,4.137508,0-4地债ETF


In [70]:
# feature
Rp = np.dot(r[funds], weight)
100*240*Rp.mean(), 100*15.5*Rp.std()

(0.007131749725057535, 4.631950479977869)